# Sentinel Vision — Demo Notebook

Automated video surveillance analytics using modern computer vision.

## Setup

Run the cell below to install dependencies and clone the repo.

In [ ]:
from pathlib import Path
from google.colab import files

DEFAULT_VIDEO = "The CCTV People Demo 2.mp4"

# If no video found, prompt user to upload
if not Path(DEFAULT_VIDEO).exists():
    print(f"No default video found. Please upload a CCTV video file.")

uploaded = files.upload()
if uploaded:
    input_video = list(uploaded.keys())[0]
    print(f"Using uploaded: {input_video}")
elif (Path(DEFAULT_VIDEO).exists()):
    input_video = DEFAULT_VIDEO
    print(f"Using default: {input_video}")
else:
    raise FileNotFoundError("No video file provided.")


## Download or Upload Video

Choose a test video or upload your own CCTV footage.

[Default test video](https://www.youtube.com/watch?v=GJNjaRJWVP8) (1080p, downloaded via yt-dlp)

In [ ]:
from pathlib import Path
from google.colab import files

DEFAULT_VIDEO = "The CCTV People Demo 2.mp4"
DEFAULT_URL = "https://www.youtube.com/watch?v=GJNjaRJWVP8"

if not Path(DEFAULT_VIDEO).exists():
    print(f"Downloading 1080p test video...")
    !yt-dlp -f 'bestvideo[height<=1080]+bestaudio/best[height<=1080]' \
        --merge-output-format mp4 -o "{DEFAULT_VIDEO}" "{DEFAULT_URL}"

uploaded = files.upload()
if uploaded:
    input_video = list(uploaded.keys())[0]
    print(f"Using uploaded: {input_video}")
elif (Path(DEFAULT_VIDEO).exists()):
    input_video = DEFAULT_VIDEO
    print(f"Using downloaded default: {input_video}")


## Run Analysis

This processes the video with detection, tracking, analytics, and generates all outputs.

In [ ]:
# Pull latest Phase 3.5 code & reload modules
!git checkout phase-3-5-advanced-intelligence 2>/dev/null; git pull
%load_ext autoreload
%autoreload 2

import torch
from src.pipeline import analyze_video
import json

zone_config = json.load(open("configs/demo_zones.json"))
calibration_config = json.load(open("configs/demo_calibration.json"))

result = analyze_video(
    video_path=input_video,
    output_dir="outputs",
    model_size="xlarge",
    conf_threshold=0.25,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_reid=True,
    track_thresh=0.5,
    match_thresh=0.8,
    track_buffer=300,
    zone_config=zone_config,
    calibration_config=calibration_config,
    capture_evidence=True,
)

## View Summary Report

In [ ]:
print(open("outputs/summary.txt").read())

## Download Outputs

Download the annotated video and analytics data.

In [ ]:
from google.colab import files

files.download("outputs/output_tracking.mp4")
files.download("outputs/analytics.json")
files.download("outputs/summary.txt")